# Linking

**Goal:** join detections from the 5 frames of a night into straight-line **tracklets**, and measure how many planted
asteroids come out as clean tracklets. This is a **search** problem, not a learning problem: physics (straight lines,
plausible speeds) does the work, and the CNN keeps the search small.

Read `docs/linking.md` alongside this notebook.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.evaluate.choose_threshold import choose_threshold
from asthunt.link.link_detections_into_tracklets import link_detections_into_tracklets
from asthunt.label.label_tracklets import label_tracklets
from asthunt.dataset.build_tracklet_dataset import build_tracklet_dataset
from asthunt.dataset.assemble_stamp_dataset import assemble_stamp_dataset
from asthunt.detect.cut_triplet_stamps import cut_triplet_stamps

project = load_config("project"); paths = project["paths"]
acquire, link = load_config("acquire"), load_config("link")


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


nights, field = acquire["nights"], acquire["field"]


def load_night(night):
    tag = make_unit_tag(unit_of(night))
    detections = pd.read_parquet(paths["processed"] / f"detections__{tag}.parquet")
    detections["score"] = np.load(paths["processed"] / f"realbogus_scores__{tag}.npy")
    # The pointing shifts between exposures: link on frame 0's common grid, not each frame's own pixels
    detections["x_frame"], detections["y_frame"] = detections["x"], detections["y"]
    detections["x"], detections["y"] = detections["x_common"], detections["y_common"]
    return detections

night = nights["train"][0]
detections = load_night(night)
print(night, len(detections), "detections")

## 1. Why we can't try everything
One detection per frame, every combination:

In [ ]:
per_frame = detections.groupby("frame_index").size()
print(per_frame.to_string())
print(f"all combinations of one detection per frame: {np.prod(per_frame.to_numpy(dtype=float)):.1e}")

Instead: **seed and predict**. Two detections in different frames at a plausible speed define a straight line;
we only *look up* where the other detections should be. A KD-tree makes each look-up nearly instant.

How many seed pairs are there between the first and last frame, before and after the CNN filter?

In [ ]:
from scipy.spatial import cKDTree

def count_seeds(rows, first, last):
    a, b = rows[rows["frame_index"] == first], rows[rows["frame_index"] == last]
    hours = (b["obsjd"].iloc[0] - a["obsjd"].iloc[0]) * 24
    max_distance = link["max_rate_arcsec_per_hour"] * hours / link["pixel_scale_arcsec"]
    neighbours = cKDTree(b[["x", "y"]].to_numpy()).query_ball_point(a[["x", "y"]].to_numpy(), r=max_distance)
    return sum(len(n) for n in neighbours)

print("seed pairs, frames 0 -> 4, all detections:", count_seeds(detections, 0, 4))

## 2. The CNN threshold for linking
Linking needs high **recall**: a detection thrown away here can never be part of a tracklet. We keep 95% of real
detections, choosing the threshold on **validation** nights (final CNN's scores).

In [ ]:
validation = pd.concat([load_night(n) for n in nights["validation"]], ignore_index=True)
realbogus_threshold = choose_threshold(validation["label"], validation["score"], target_recall=0.95)
kept = detections[detections["score"] >= realbogus_threshold]
print(f"threshold {realbogus_threshold:.3f}: keeps {len(kept)} of {len(detections)} detections ({len(kept)/len(detections):.1%})")
print("seed pairs, frames 0 -> 4, after the CNN filter:", count_seeds(kept, 0, 4))

## 3. Link one night

In [ ]:
start = time.time()
tracklets, members = link_detections_into_tracklets(
    kept, pixel_scale=link["pixel_scale_arcsec"], min_rate=link["min_rate_arcsec_per_hour"],
    max_rate=link["max_rate_arcsec_per_hour"], match_radius=link["match_radius_pixels"],
    min_detections=link["min_detections"], max_residual=link["max_residual_pixels"])
tracklets = label_tracklets(tracklets, members, detections)
print(f"{len(tracklets)} tracklets in {time.time() - start:.1f} s: {tracklets['label'].sum()} real, {(tracklets['label'] == 0).sum()} bogus")
tracklets.head()

## 4. Linking completeness
Of the fakes that *could* be linked (detected and kept in at least 4 frames), how many became a clean tracklet?

In [ ]:
injections = pd.read_parquet(paths["processed"] / f"injections__{make_unit_tag(unit_of(night))}.parquet")
frames_kept = kept[kept["injection_id"] >= 0].groupby("injection_id")["frame_index"].nunique()
speed = injections.groupby("injection_id")["rate_arcsec_per_hour"].first()
in_speed_range = speed.between(link["min_rate_arcsec_per_hour"], link["max_rate_arcsec_per_hour"])
# linkable = kept in enough frames AND moving at a speed the linker searches for
linkable = set(frames_kept[frames_kept >= link["min_detections"]].index) & set(speed[in_speed_range].index)
print(f"fakes slower than {link['min_rate_arcsec_per_hour']} \"/hr (excluded by design): {(~in_speed_range).sum()}")
linked = set(tracklets.loc[tracklets["label"] == 1, "injection_id"])

per_fake = injections.groupby("injection_id").agg(mag=("mag", "first"), rate=("rate_arcsec_per_hour", "first"))
per_fake["linkable"] = per_fake.index.isin(linkable)
per_fake["linked"] = per_fake.index.isin(linked)
print(f"fakes planted: {len(per_fake)}, linkable: {per_fake['linkable'].sum()}, linked: {per_fake['linked'].sum()}")
linked_and_linkable = (per_fake["linked"] & per_fake["linkable"]).sum()
print(f"linking efficiency (linked among linkable): {linked_and_linkable / per_fake['linkable'].sum():.1%}")
print(f"linked although planted just below the speed limit (measurement noise): {(per_fake['linked'] & ~per_fake['linkable']).sum()}")

bins = np.arange(17, 22.5, 0.5)
grouped = per_fake.groupby(pd.cut(per_fake["mag"], bins), observed=True)[["linkable", "linked"]].mean()
plt.plot([i.mid for i in grouped.index], grouped["linkable"], "o--", label="linkable (>=4 frames kept, speed in range)")
plt.plot([i.mid for i in grouped.index], grouped["linked"], "s-", label="linked as a clean tracklet")
plt.xlabel("magnitude"); plt.ylabel("fraction of planted fakes"); plt.ylim(0, 1); plt.legend(); plt.show()

## 5. What do the tracklets look like?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for label, colour, name in [(0, "tab:red", "bogus"), (1, "tab:blue", "real")]:
    for i, tracklet_id in enumerate(tracklets.loc[tracklets["label"] == label, "tracklet_id"]):
        ids = members.loc[members["tracklet_id"] == tracklet_id, "detection_id"]
        rows = detections.loc[ids].sort_values("obsjd")
        ax.plot(rows["x"], rows["y"], "-", c=colour, lw=1.5, label=name if i == 0 else None)
ax.set_aspect("equal"); ax.legend(); ax.set_title("every tracklet on the chip"); plt.show()

print("bogus tracklets by composition:")
joined = members.merge(detections[["injection_id"]], left_on="detection_id", right_index=True)
composition = joined.groupby("tracklet_id")["injection_id"].agg(lambda s: "fake + noise" if (s >= 0).any() else "noise only")
print(composition[tracklets.set_index("tracklet_id")["label"] == 0].value_counts().to_string())

Two kinds of bogus tracklets:
- **noise only:** 4+ unrelated detections that happen to line up. These are chance alignments.
- **fake + noise:** a real track that borrowed a wrong detection, or a partial track. These **near misses** are the
  hard negatives the *Leakage audit* predicted. Now they come from the real linker, not from our imagination.

## 6. Build tracklets for every night (for the tracklet classifier)
Training nights use **out-of-fold** CNN scores; validation and test nights use the final model's scores.

In [ ]:
summary = []
for split, split_nights in nights.items():
    for n in split_nights:
        start = time.time()
        t, m = build_tracklet_dataset(unit_of(n), paths["processed"], realbogus_threshold, link)
        summary.append({"split": split, "night": n, "tracklets": len(t), "real": int(t["label"].sum()),
                        "bogus": int((t["label"] == 0).sum()), "seconds": round(time.time() - start, 1)})
pd.DataFrame(summary)

In [ ]:
import json
# Update (don't overwrite): other phases store their own thresholds in the same file
thresholds_file = paths["processed"] / "thresholds.json"
thresholds = json.loads(thresholds_file.read_text()) if thresholds_file.exists() else {}
thresholds["realbogus"] = realbogus_threshold
thresholds_file.write_text(json.dumps(thresholds, indent=1))
print("saved realbogus threshold", round(realbogus_threshold, 4))

## Exercises
1. Set `min_detections` to 3. How many more real tracklets, and how many more bogus ones?
2. Double `match_radius_pixels`. What happens to the number of "fake + noise" tracklets? Why?
3. Lower the CNN threshold to keep 99% of real detections. How long does linking take now?